# RAG-Based Intelligent Document QA System
**Mistral-7B-Instruct + FAISS Dense + BM25 Sparse + Reciprocal Rank Fusion**

Author: Yuvraj Chulpar | NIT Kurukshetra | [GitHub](https://github.com/chulparyuvraj/rag-document-qa)

---
### Pipeline Overview
1. PDF Ingestion → Chunking (512 tokens, 64 overlap)
2. FAISS dense index (sentence-transformers/all-MiniLM-L6-v2)
3. BM25 sparse index (rank_bm25)
4. Hybrid retrieval with Reciprocal Rank Fusion (RRF)
5. Mistral-7B-Instruct-v0.2 in 4-bit QLoRA for answer generation

> ⚠️ **Run Cell 1 first, then restart the kernel, then run all remaining cells in order.**


## Cell 1 — Install Dependencies
> ⚠️ After this cell finishes, **restart the kernel**, then run from Cell 2 onwards.

In [ ]:
# Fix numpy FIRST — must be upgraded before any other package
import subprocess
subprocess.run('pip install -q "numpy>=1.26.0" --upgrade', shell=True)

# Install all required packages
!pip install -q \
    pymupdf loguru tqdm rank_bm25 \
    langchain-core langchain-community langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers faiss-cpu \
    transformers peft trl bitsandbytes accelerate datasets \
    rouge-score python-dotenv huggingface_hub

print("✅ All packages installed — now RESTART THE KERNEL then run Cell 2 onwards")


## Cell 2 — Project Setup

In [ ]:
import sys, os, shutil, subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr and 'error' in result.stderr.lower(): print("STDERR:", result.stderr[:300])

# Verify numpy is correct version
import numpy as np
print(f"numpy version: {np.__version__}")
assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (1, 26), \
    "numpy too old — restart kernel and rerun from Cell 1"
print("✅ numpy OK")

# Copy project to writable working directory
BASE = '/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa'
WORK = '/kaggle/working/rag-document-qa'

if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(BASE, WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(f"✅ Working directory: {os.getcwd()}")


numpy version: 2.0.2
✅ numpy OK
✅ Working directory: /kaggle/working/rag-document-qa


## Cell 3 — Fix Broken Files

In [ ]:
# Remove the brace-expanded broken __init__.py
broken = os.path.join(WORK, "src/{ingestion,embeddings,retrieval,generation,finetuning,api}/__init__.py")
if os.path.exists(broken):
    os.remove(broken)
    print("Removed broken __init__.py")

# Ensure all __init__.py exist
for mod in ['src','src/ingestion','src/retrieval','src/pipeline','src/finetuning','src/api']:
    init = f'{WORK}/{mod}/__init__.py'
    os.makedirs(os.path.dirname(init), exist_ok=True)
    if not os.path.exists(init):
        open(init, 'w').close()
        print(f"Created: {mod}/__init__.py")

# Patch all broken LangChain imports across src/
fixes = {
    'from langchain.text_splitter import RecursiveCharacterTextSplitter':
        'from langchain_text_splitters import RecursiveCharacterTextSplitter',
    'from langchain.schema import Document':
        'from langchain_core.documents import Document',
    'from langchain.schema.retriever import BaseRetriever':
        'from langchain_core.retrievers import BaseRetriever',
    'from langchain.callbacks.manager import CallbackManagerForRetrieverRun':
        'from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun',
    'from langchain.prompts import PromptTemplate':
        'from langchain_core.prompts import PromptTemplate',
    'from langchain_community.llms import HuggingFacePipeline':
        'from langchain_huggingface import HuggingFacePipeline',
    'from langchain_community.chains import RetrievalQA': '',
    'from langchain.chains import RetrievalQA': '',
}

for root, dirs, files in os.walk('src'):
    for fname in files:
        if not fname.endswith('.py'): continue
        fpath = os.path.join(root, fname)
        with open(fpath, 'r') as f: content = f.read()
        new_content = content
        for old, new in fixes.items():
            new_content = new_content.replace(old, new)
        if new_content != content:
            with open(fpath, 'w') as f: f.write(new_content)
            print(f"✅ Patched: {fpath}")

print("✅ All imports fixed")


✅ Patched: src/ingestion/chunker.py
✅ Patched: src/pipeline/prompt_templates.py
✅ All imports fixed


## Cell 4 — Rewrite Retriever Files (LangChain v0.2 Compatible)

In [ ]:
# Rewrite bm25_retriever.py
bm25_code = '''
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from rank_bm25 import BM25Okapi
from loguru import logger
import re


class BM25Retriever(BaseRetriever):
    documents: List[Document]
    bm25: object
    k: int = 5

    class Config:
        arbitrary_types_allowed = True

    @classmethod
    def from_documents(cls, documents: List[Document], k: int = 5) -> "BM25Retriever":
        tokenized_corpus = [cls._tokenize(doc.page_content) for doc in documents]
        bm25 = BM25Okapi(tokenized_corpus)
        logger.info(f"BM25 index built on {len(documents)} documents")
        return cls(documents=documents, bm25=bm25, k=k)

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_k_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:self.k]
        results = []
        for idx in top_k_indices:
            doc = self.documents[idx]
            doc.metadata["bm25_score"] = float(scores[idx])
            results.append(doc)
        return results

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.findall(r"[a-z0-9]+", text.lower())
'''

with open('src/retrieval/bm25_retriever.py', 'w') as f:
    f.write(bm25_code)
print("✅ bm25_retriever.py rewritten")

# Rewrite hybrid_retriever.py
hybrid_code = '''
from typing import List, Dict
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from loguru import logger

from .vector_store import FAISSVectorStore
from .bm25_retriever import BM25Retriever


class HybridRetriever(BaseRetriever):
    faiss_store: FAISSVectorStore
    bm25_retriever: BM25Retriever
    k: int = 5
    rrf_k: int = 60
    dense_weight: float = 0.6
    sparse_weight: float = 0.4

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        fetch_k = self.k * 3
        dense_docs  = self.faiss_store.similarity_search(query, k=fetch_k)
        sparse_docs = self.bm25_retriever._get_relevant_documents(query)
        fused = self._rrf(dense_docs, sparse_docs)
        logger.debug(f"Hybrid: dense={len(dense_docs)}, sparse={len(sparse_docs)}, returning top {self.k}")
        return fused[:self.k]

    def _rrf(self, dense_docs: List[Document], sparse_docs: List[Document]) -> List[Document]:
        scores: Dict[str, float] = {}
        doc_map: Dict[str, Document] = {}
        for rank, doc in enumerate(dense_docs):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + self.dense_weight * (1.0 / (self.rrf_k + rank + 1))
            doc_map[key] = doc
        for rank, doc in enumerate(sparse_docs):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + self.sparse_weight * (1.0 / (self.rrf_k + rank + 1))
            doc_map[key] = doc
        sorted_keys = sorted(scores, key=scores.__getitem__, reverse=True)
        results = []
        for key in sorted_keys:
            doc = doc_map[key]
            doc.metadata["rrf_score"] = round(scores[key], 6)
            results.append(doc)
        return results
'''

with open('src/retrieval/hybrid_retriever.py', 'w') as f:
    f.write(hybrid_code)
print("✅ hybrid_retriever.py rewritten")

# Rewrite rag_chain.py
rag_chain_code = '''
import os
from typing import Optional, List
from pathlib import Path

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_huggingface import HuggingFacePipeline
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig,
)
from peft import PeftModel
import torch
from loguru import logger

from .prompt_templates import RAG_PROMPT
from ..ingestion.pdf_loader import PDFLoader
from ..ingestion.chunker import ResearchPaperChunker
from ..retrieval.embeddings import get_embedding_model
from ..retrieval.vector_store import FAISSVectorStore
from ..retrieval.bm25_retriever import BM25Retriever
from ..retrieval.hybrid_retriever import HybridRetriever


class RAGPipeline:
    def __init__(
        self,
        embedding_model_name: str = None,
        llm_model_name: str = None,
        adapter_path: Optional[str] = None,
        index_path: str = "indexes/faiss",
        device: str = "auto",
        top_k: int = 5,
    ):
        self.embedding_model_name = embedding_model_name or os.getenv(
            "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
        )
        self.llm_model_name = llm_model_name or os.getenv(
            "BASE_LLM", "mistralai/Mistral-7B-Instruct-v0.2"
        )
        self.adapter_path = adapter_path or os.getenv("FINETUNED_ADAPTER_PATH")
        self.index_path = index_path
        self.device = device
        self.top_k = top_k
        self._embeddings       = None
        self._faiss_store      = None
        self._bm25             = None
        self._hybrid_retriever = None
        self._llm_pipeline     = None
        self._chunks: List[Document] = []

    def build_index(self, pdf_dir="data/papers", save_index=True, force_rebuild=False):
        self._embeddings  = get_embedding_model(self.embedding_model_name)
        self._faiss_store = FAISSVectorStore(self._embeddings)

        if not force_rebuild and Path(self.index_path).exists():
            logger.info("Loading existing FAISS index from disk...")
            self._faiss_store.load(self.index_path)
            self._chunks = list(self._faiss_store._index.docstore._dict.values())
        else:
            logger.info("Building new index from PDFs...")
            loader  = PDFLoader(source_dir=pdf_dir)
            pages   = loader.load_all()
            chunker = ResearchPaperChunker(chunk_size=512, chunk_overlap=64)
            self._chunks = chunker.chunk(pages)
            self._faiss_store.build(self._chunks)
            if save_index:
                self._faiss_store.save(self.index_path)

        self._bm25             = BM25Retriever.from_documents(self._chunks, k=self.top_k)
        self._hybrid_retriever = HybridRetriever(
            faiss_store=self._faiss_store,
            bm25_retriever=self._bm25,
            k=self.top_k,
        )
        self._llm_pipeline = self._load_llm()
        logger.info("RAG pipeline ready!")

    def query(self, question: str) -> dict:
        if self._llm_pipeline is None:
            raise RuntimeError("Call build_index() first.")
        logger.info(f"Query: {question}")
        source_docs  = self._hybrid_retriever._get_relevant_documents(question)
        context      = "\\n\\n".join([doc.page_content for doc in source_docs])
        prompt_text  = RAG_PROMPT.format(context=context, question=question)
        response     = self._llm_pipeline.pipeline(
            prompt_text, max_new_tokens=512, temperature=0.1,
            do_sample=True, repetition_penalty=1.15,
        )
        answer = response[0]["generated_text"][len(prompt_text):].strip()
        return {"result": answer, "source_documents": source_docs}

    def get_context(self, question: str) -> List[Document]:
        return self._hybrid_retriever._get_relevant_documents(question)

    def _load_llm(self) -> HuggingFacePipeline:
        logger.info(f"Loading LLM: {self.llm_model_name}")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.llm_model_name)
        tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            self.llm_model_name,
            quantization_config=bnb_config,
            device_map=self.device,
            trust_remote_code=True,
        )
        if self.adapter_path and Path(self.adapter_path).exists():
            logger.info(f"Loading QLoRA adapter from {self.adapter_path}")
            model = PeftModel.from_pretrained(model, self.adapter_path)
            model = model.merge_and_unload()
        pipe = pipeline(
            "text-generation", model=model, tokenizer=tokenizer,
            max_new_tokens=512, temperature=0.1, do_sample=True,
            repetition_penalty=1.15,
        )
        return HuggingFacePipeline(pipeline=pipe)
'''

with open('src/pipeline/rag_chain.py', 'w') as f:
    f.write(rag_chain_code)
print("✅ rag_chain.py rewritten")


✅ bm25_retriever.py rewritten
✅ hybrid_retriever.py rewritten
✅ rag_chain.py rewritten


## Cell 5 — Verify All Imports

In [ ]:
from src.ingestion.pdf_loader import PDFLoader
from src.ingestion.chunker import ResearchPaperChunker
from src.retrieval.embeddings import get_embedding_model
from src.retrieval.vector_store import FAISSVectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import HybridRetriever
from src.pipeline.rag_chain import RAGPipeline
from src.pipeline.prompt_templates import RAG_PROMPT

print("✅ All imports successful!")


✅ All imports successful!


## Cell 6 — HuggingFace Login
> Replace `hf_your_token_here` with your real token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
> Never share your token publicly.

In [ ]:
from huggingface_hub import login

# Paste your HuggingFace token below (Read access required)
# Get it from: https://huggingface.co/settings/tokens
HF_TOKEN = 'hf_your_token_here'

login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")


## Cell 7 — Download Research Papers
Adds arXiv papers to the index. You can add more IDs to the list.

In [ ]:
import urllib.request

os.makedirs('data/papers', exist_ok=True)

# Add any arXiv paper IDs you want to index
arxiv_ids = [
    '2512.20363',   # Clust-PSI-PFL (your FL paper)
    '1706.03762',   # Attention Is All You Need
]

for arxiv_id in arxiv_ids:
    out_path = f'data/papers/{arxiv_id}.pdf'
    if not os.path.exists(out_path):
        url = f'https://arxiv.org/pdf/{arxiv_id}.pdf'
        urllib.request.urlretrieve(url, out_path)
        print(f'✅ Downloaded: {arxiv_id}.pdf')
    else:
        print(f'⏭ Already exists: {arxiv_id}.pdf')


✅ Downloaded: 2512.20363.pdf
✅ Downloaded: 1706.03762.pdf


## Cell 8 — Build FAISS + BM25 Index
This embeds all PDF chunks using GPU-accelerated sentence-transformers.
Expected time: ~2 minutes on T4.

In [ ]:
# Load & chunk PDFs
loader  = PDFLoader(source_dir='data/papers')
pages   = loader.load_all()
print(f'Pages extracted: {len(pages)}')

chunker = ResearchPaperChunker(chunk_size=512, chunk_overlap=64)
chunks  = chunker.chunk(pages)
print(f'Chunks created:  {len(chunks)}')

# Build FAISS dense index
embedding_model = get_embedding_model(device='cuda')
faiss_store     = FAISSVectorStore(embedding_model)
faiss_store.build(chunks)
faiss_store.save('indexes/faiss')
print("✅ FAISS index saved")

# Build BM25 sparse index + Hybrid retriever
bm25   = BM25Retriever.from_documents(chunks, k=5)
hybrid = HybridRetriever(faiss_store=faiss_store, bm25_retriever=bm25, k=5)
print("✅ BM25 + Hybrid retriever ready")


Pages extracted: 29
Chunks created:  250
✅ FAISS index saved
✅ BM25 + Hybrid retriever ready


## Cell 9 — Test Hybrid Retrieval (No LLM Needed)
Verify retrieval is working before loading the 14GB model.

In [ ]:
test_query = "What is the PSI clustering method in federated learning?"
docs = hybrid._get_relevant_documents(test_query)

print(f'✅ Retrieved {len(docs)} docs for: "{test_query}"')
print()
for i, d in enumerate(docs):
    print(f"[{i+1}] {d.metadata['source']} — page {d.metadata['page']} | RRF: {d.metadata.get('rrf_score', 'N/A')}")
    print(f"    {d.page_content[:120].strip()}...")
    print()


✅ Retrieved 5 docs for: "What is the PSI clustering method in federated learning?"

[1] 2512.20363.pdf — page 3 | RRF: 0.009524
    dom selection, it achieves faster convergence...

[2] 2512.20363.pdf — page 4 | RRF: 0.008929
    PSI measures the distributional shift...



## Cell 10 — Load Mistral-7B-Instruct (4-bit QLoRA)
> ⏳ Takes **5–8 minutes** on T4. Do not interrupt.
> Requires HuggingFace login (Cell 6) and license agreement on the model page.

In [ ]:
rag = RAGPipeline(index_path='indexes/faiss', top_k=5)

# Reuse already-built index objects — skip rebuilding
rag._embeddings        = embedding_model
rag._faiss_store       = faiss_store
rag._chunks            = chunks
rag._bm25              = bm25
rag._hybrid_retriever  = hybrid

print("Loading Mistral-7B-Instruct-v0.2 in 4-bit... (~5-8 mins on T4)")
rag._llm_pipeline = rag._load_llm()
print("✅ Mistral-7B loaded and ready!")


## Cell 11 — Define ask() Helper Function

In [ ]:
def ask(question, retriever=hybrid, llm=None):
    """Query the RAG pipeline directly."""
    if llm is None:
        llm = rag._llm_pipeline

    # Retrieve relevant chunks
    source_docs = retriever._get_relevant_documents(question)

    # Build context + format prompt
    context     = "\n\n".join([doc.page_content for doc in source_docs])
    prompt_text = RAG_PROMPT.format(context=context, question=question)

    # Generate answer
    response = llm.pipeline(
        prompt_text,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        repetition_penalty=1.15,
    )
    answer = response[0]["generated_text"][len(prompt_text):].strip()
    return {"result": answer, "source_documents": source_docs}

print("✅ ask() function ready")


✅ ask() function ready


## Cell 12 — Query Your FL Paper (Clust-PSI-PFL)
Testing domain-specific QA on your research paper.

In [ ]:
questions = [
    "What is the cluster federated Learning?",
    "What datasets were used for evaluation?",
    "What are the main results of this paper?",
    "What is Population Stability Index and how is it used?",
    "What are the baseline methods compared in this paper?",
]

for q in questions:
    result = ask(q)
    print(f"Q: {q}")
    print(f"A: {result['result']}")
    sources = [f"{d.metadata['source']} p.{d.metadata['page']}" for d in result['source_documents']]
    print(f"Sources: {sources}")
    print("-" * 70)


Q: What is the cluster federated Learning?
A: Cluster Federated Learning (CFL) is a method for handling non-IID data in federated learning. It recursively clusters clients via the cosine similarity of their local updates when training stalls, aiming to minimize cross-cluster similarity (Sattler et al., 2020).
Sources: ['2512.20363.pdf p.12', '2512.20363.pdf p.3', '2512.20363.pdf p.3']
----------------------------------------------------------------------
Q: What datasets were used for evaluation?
A: The evaluation was conducted on six datasets: ACSIncome, Serengeti, FMNIST, CIFAR10, Sent140, and Amazon reviews (Refer Table I on pages 23-24 for detailed information).
Sources: ['2512.20363.pdf p.13', '2512.20363.pdf p.7', '2512.20363.pdf p.6']
----------------------------------------------------------------------
Q: What are the main results of this paper?
A: The proposed Clust-PSI-PFL method achieves up to 18% relative gains over strong baselines such as CFL and FedSoft across six datas

## Cell 13 — Generate QA Pairs for Fine-tuning
Auto-generates training data from your FL paper chunks using Mistral.
Used in Notebook 02 for QLoRA fine-tuning.

In [ ]:
import json, re
from tqdm import tqdm

def generate_qa_from_chunk(chunk_text, llm_pipeline):
    prompt = f"""<s>[INST] Read the following text from a research paper on Federated Learning \
and generate exactly ONE question and answer pair.

Text:
{chunk_text[:600]}

Respond in this exact format:
QUESTION: <your question here>
ANSWER: <your answer here>

Rules:
- Question must be specific and answerable from the text
- Answer must be grounded in the text only
- Do not add any other text [/INST]"""

    response = llm_pipeline.pipeline(
        prompt, max_new_tokens=200, temperature=0.3,
        do_sample=True, repetition_penalty=1.1,
    )
    generated = response[0]["generated_text"][len(prompt):].strip()
    q_match = re.search(r'QUESTION:\s*(.+?)(?=ANSWER:|$)', generated, re.DOTALL)
    a_match = re.search(r'ANSWER:\s*(.+?)$', generated, re.DOTALL)
    if q_match and a_match:
        return {
            "context":  chunk_text[:600],
            "question": q_match.group(1).strip(),
            "answer":   a_match.group(1).strip()
        }
    return None

# Use only FL paper chunks for domain-specific fine-tuning
fl_chunks = [c for c in chunks if '2512.20363' in c.metadata.get('source', '')]
print(f"FL paper chunks: {len(fl_chunks)}")

# Sample every 2nd chunk for diversity (max 60 → ~50 valid pairs)
selected = fl_chunks[::2][:60]
print(f"Generating QA pairs from {len(selected)} chunks...")

qa_pairs = []
for chunk in tqdm(selected):
    pair = generate_qa_from_chunk(chunk.page_content, rag._llm_pipeline)
    if pair:
        qa_pairs.append(pair)

print(f"\n✅ Generated {len(qa_pairs)} QA pairs")
if qa_pairs:
    print(f"\nSample:")
    print(f"Q: {qa_pairs[0]['question']}")
    print(f"A: {qa_pairs[0]['answer'][:200]}...")

# Save to disk
os.makedirs('data', exist_ok=True)
with open('data/fl_qa_pairs.json', 'w') as f:
    json.dump(qa_pairs, f, indent=2)
print(f"\n✅ Saved to data/fl_qa_pairs.json")


---
## Results Summary

| Metric | Value |
|--------|-------|
| Papers indexed | 2 (Clust-PSI-PFL + Attention Is All You Need) |
| Total pages | 29 |
| Chunks | 250 |
| Embedding model | sentence-transformers/all-MiniLM-L6-v2 |
| LLM | Mistral-7B-Instruct-v0.2 (4-bit QLoRA) |
| Retrieval | FAISS dense + BM25 sparse + RRF fusion |
| Docs retrieved per query | 5 |

**Next:** Run `02_QLoRA_Finetuning.ipynb` to fine-tune Mistral on the generated QA pairs for improved domain accuracy.
